In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
TRAIN_PATH = "../data/train.csv"

df = pd.read_csv(TRAIN_PATH)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

display(df.head())

Shape: (57477, 9)

Columns:
['id', 'model_a', 'model_b', 'prompt', 'response_a', 'response_b', 'winner_model_a', 'winner_model_b', 'winner_tie']


,id,model_a,model_b,prompt,response_a,response_b,winner_model_a,winner_model_b,winner_tie
0,30192,gpt-4-1106-preview,gpt-4-0613,"[""Is it morally right to try to have a certain...","[""The question of whether it is morally right ...","[""As an AI, I don't have personal beliefs or o...",1,0,0
1,53567,koala-13b,gpt-4-0613,"[""What is the difference between marriage lice...","[""A marriage license is a legal document that ...","[""A marriage license and a marriage certificat...",0,1,0
2,65089,gpt-3.5-turbo-0613,mistral-medium,"[""explain function calling. how would you call...","[""Function calling is the process of invoking ...","[""Function calling is the process of invoking ...",0,0,1
3,96401,llama-2-13b-chat,mistral-7b-instruct,"[""How can I create a test set for a very rare ...","[""Creating a test set for a very rare category...","[""When building a classifier for a very rare c...",1,0,0
4,198779,koala-13b,gpt-3.5-turbo-0314,"[""What is the best way to travel from Tel-Aviv...","[""The best way to travel from Tel Aviv to Jeru...","[""The best way to travel from Tel-Aviv to Jeru...",0,1,0


In [3]:
print(df.loc[0, "prompt"])
print()
print(df.loc[0, "response_a"])
print()
print(df.loc[0, "response_b"])

["Is it morally right to try to have a certain percentage of females on managerial positions?","OK, does pineapple belong on a pizza? Relax and give me fun answer."]

["The question of whether it is morally right to aim for a certain percentage of females in managerial positions is a complex ethical issue that involves considerations of fairness, equality, diversity, and discrimination.\n\nHere are some arguments in favor of and against such policies:\n\n**Arguments in favor:**\n\n1. **Correcting Historical Inequities:** Women have historically been underrepresented in leadership roles due to various cultural, institutional, and social barriers. Aiming for a specific percentage can be seen as a corrective measure to address past and ongoing discrimination.\n\n2. **Promoting Diversity:** Diverse leadership teams can enhance decision-making and represent a broader range of perspectives. This can lead to better outcomes for organizations and society as a whole.\n\n3. **Equality of Opportu

In [4]:
import json

prompt = json.loads(df.loc[0, "prompt"])
response_a = json.loads(df.loc[0, "response_a"])
response_b = json.loads(df.loc[0, "response_b"])

print(prompt[1])
print(response_a)
print(response_b)

OK, does pineapple belong on a pizza? Relax and give me fun answer.
['The question of whether it is morally right to aim for a certain percentage of females in managerial positions is a complex ethical issue that involves considerations of fairness, equality, diversity, and discrimination.\n\nHere are some arguments in favor of and against such policies:\n\n**Arguments in favor:**\n\n1. **Correcting Historical Inequities:** Women have historically been underrepresented in leadership roles due to various cultural, institutional, and social barriers. Aiming for a specific percentage can be seen as a corrective measure to address past and ongoing discrimination.\n\n2. **Promoting Diversity:** Diverse leadership teams can enhance decision-making and represent a broader range of perspectives. This can lead to better outcomes for organizations and society as a whole.\n\n3. **Equality of Opportunity:** Setting targets for female representation in management can help ensure that women have equ

In [5]:
df["prompt_parsed"] = df["prompt"].apply(json.loads)
df["response_a_parsed"] = df["response_a"].apply(json.loads)
df["response_b_parsed"] = df["response_b"].apply(json.loads)

In [6]:
# print(df["prompt_parsed"].apply(len).value_counts().sort_index())
# print(df["response_a_parsed"].apply(len).value_counts().sort_index())
# print(df["response_b_parsed"].apply(len).value_counts().sort_index())
assert all(
    len(p) == len(a) == len(b)
    for p, a, b in zip(
        df["prompt_parsed"],
        df["response_a_parsed"],
        df["response_b_parsed"]
    )
)

print("All conversations are aligned!")

All conversations are aligned!


In [7]:
target_cols = [
    "winner_model_a",
    "winner_model_b",
    "winner_tie"
]

df["label"] = df[target_cols].values.argmax(axis=1)
print(df["label"].value_counts().sort_index())
print()
print((df["label"].value_counts(normalize=True).sort_index() * 100).round(2))

label
0    20064
1    19652
2    17761
Name: count, dtype: int64

label
0    34.91
1    34.19
2    30.90
Name: proportion, dtype: float64


In [8]:
def flatten(x):
    return "\n".join(str(item) for item in x if item is not None)

df["prompt_text"] = df["prompt_parsed"].apply(flatten)
df["response_a_text"] = df["response_a_parsed"].apply(flatten)
df["response_b_text"] = df["response_b_parsed"].apply(flatten)

In [9]:
# for col in ["prompt_text", "response_a_text", "response_b_text"]:
#     df[col + "_words"] = df[col].str.split().str.len()

# df[
#     ["prompt_text_words", "response_a_text_words", "response_b_text_words"]
# ].describe()
df["prompt_text"].loc[0]
df["response_a_text"].loc[0]


'The question of whether it is morally right to aim for a certain percentage of females in managerial positions is a complex ethical issue that involves considerations of fairness, equality, diversity, and discrimination.\n\nHere are some arguments in favor of and against such policies:\n\n**Arguments in favor:**\n\n1. **Correcting Historical Inequities:** Women have historically been underrepresented in leadership roles due to various cultural, institutional, and social barriers. Aiming for a specific percentage can be seen as a corrective measure to address past and ongoing discrimination.\n\n2. **Promoting Diversity:** Diverse leadership teams can enhance decision-making and represent a broader range of perspectives. This can lead to better outcomes for organizations and society as a whole.\n\n3. **Equality of Opportunity:** Setting targets for female representation in management can help ensure that women have equal opportunities to advance in their careers.\n\n4. **Role Modeling:*

In [10]:
def build_input(row):
    conversation = []

    for prompt, a, b in zip(
        row["prompt_parsed"],
        row["response_a_parsed"],
        row["response_b_parsed"]
    ):
        conversation.append(f"User: {prompt}")
        conversation.append(f"Response A: {a}")
        conversation.append(f"Response B: {b}")

    return "\n".join(conversation)

In [11]:
df["input_text"] = df.apply(build_input, axis=1)
print(df["input_text"].iloc[0])

User: Is it morally right to try to have a certain percentage of females on managerial positions?
Response A: The question of whether it is morally right to aim for a certain percentage of females in managerial positions is a complex ethical issue that involves considerations of fairness, equality, diversity, and discrimination.

Here are some arguments in favor of and against such policies:

**Arguments in favor:**

1. **Correcting Historical Inequities:** Women have historically been underrepresented in leadership roles due to various cultural, institutional, and social barriers. Aiming for a specific percentage can be seen as a corrective measure to address past and ongoing discrimination.

2. **Promoting Diversity:** Diverse leadership teams can enhance decision-making and represent a broader range of perspectives. This can lead to better outcomes for organizations and society as a whole.

3. **Equality of Opportunity:** Setting targets for female representation in management can h

In [12]:
from transformers import AutoTokenizer

MODEL_NAME = "microsoft/deberta-v3-small"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

/Users/feiyang/miniconda3/envs/general/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
sample_lengths = []

for text in df["input_text"]:
    tokens = tokenizer(
        text,
        truncation=False
    )
    sample_lengths.append(len(tokens["input_ids"]))

df["token_length"] = sample_lengths

In [16]:
df["token_length"].describe(
    percentiles=[0.5, 0.75, 0.90, 0.95, 0.99]
)
for limit in [256, 384, 512]:
    percentage = (df["token_length"] > limit).mean() * 100
    print(f"{limit} tokens: {percentage:.2f}% truncated")

256 tokens: 76.28% truncated
384 tokens: 63.58% truncated
512 tokens: 51.35% truncated


In [17]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df,
    test_size=0.1,
    random_state=42,
    stratify=df["label"]
)

print("Train:", len(train_df))
print("Validation:", len(val_df))

Train: 51729
Validation: 5748


In [18]:
train_small = train_df.sample(1000, random_state=42).reset_index(drop=True)
val_small = val_df.sample(500, random_state=42).reset_index(drop=True)

print(len(train_small), len(val_small))

1000 500


In [19]:
MAX_LENGTH = 512

train_encodings = tokenizer(
    train_small["input_text"].tolist(),
    truncation=True,
    max_length=MAX_LENGTH
)

val_encodings = tokenizer(
    val_small["input_text"].tolist(),
    truncation=True,
    max_length=MAX_LENGTH
)

In [20]:
import torch
from torch.utils.data import Dataset

class PreferenceDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }
        item["labels"] = torch.tensor(self.labels[idx])

        return item

In [21]:
train_dataset = PreferenceDataset(
    train_encodings,
    train_small["label"]
)

val_dataset = PreferenceDataset(
    val_encodings,
    val_small["label"]
)

In [22]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3
)

Loading weights: 100%|██████████| 102/102 [00:00<00:00, 23699.26it/s]
[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING  

In [23]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.to(device)

print(device)

cpu


In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="../models/deberta-test",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    fp16=torch.cuda.is_available(),
    logging_steps=20,
    report_to="none",
)

In [27]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
)

In [28]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.000000,nan


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.05it/s]


TrainOutput(global_step=125, training_loss=2607683.584, metrics={'train_runtime': 358.3555, 'train_samples_per_second': 2.791, 'train_steps_per_second': 0.349, 'total_flos': 122670850645980.0, 'train_loss': 2607683.584, 'epoch': 1.0})

In [29]:
predictions = trainer.predict(val_dataset)

print("Predictions shape:", predictions.predictions.shape)
print("Labels shape:", predictions.label_ids.shape)
print("Predictions:")
print(predictions.predictions[:5])

/Users/feiyang/miniconda3/envs/general/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Predictions shape: (500, 3)
Labels shape: (500,)
Predictions:
[[nan nan nan]
 [nan nan nan]
 [nan nan nan]
 [nan nan nan]
 [nan nan nan]]


In [31]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3
)

print(model.classifier.weight.abs().max())

Loading weights: 100%|██████████| 102/102 [00:00<00:00, 44136.90it/s]
[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING  

tensor(0.0746, dtype=torch.float16, grad_fn=<MaxBackward1>)


In [32]:
test_output = model(**{
    k: v.unsqueeze(0)
    for k, v in train_dataset[0].items()
    if k != "labels"
})

print(test_output.logits)

tensor([[ 0.0197, -0.0066, -0.0530]], dtype=torch.float16,
       grad_fn=<AddmmBackward0>)
